In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic import hub
from langchain_core.prompts import ChatPromptTemplate
from typing import List, TypedDict
import fitz  # PyMuPDF
from uuid import uuid4
from langchain_core.documents import Document

In [ ]:
# text-embedding-004 adalah model embedding terbaru dari Google
# menghasilkan vector 768 dimensi per teks

os.environ["GOOGLE_API_KEY"] = ""

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", output_dimensionality=768)

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Mengekstrak teks dari PDF menggunakan PyMuPDF"""
    doc = fitz.open(pdf_path)
    text = "\n".join([page.get_text() for page in doc])
    return text

# Ganti path ini dengan PDF kamu
pdf_text = extract_text_from_pdf("Data_Mam_Sedap.pdf")
print(f"Total karakter: {len(pdf_text)}")
print("\nPreview 500 karakter pertama:")
print(pdf_text[:1000])

In [ ]:
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_text(pdf_text)
print(f"Total chunks: {len(chunks)}")

In [ ]:
# Preview setiap chunk (100 karakter pertama saja)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk[:100]}...")

In [ ]:
# Inisialisasi FAISS baru untuk dokumen PDF ini
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
# Bungkus setiap chunk sebagai Document object
documents = [Document(page_content=chunk) for chunk in chunks]

# Generate UUID untuk setiap dokumen
uuids = [str(uuid4()) for _ in range(len(documents))]

# Embed dan simpan ke FAISS
vector_store.add_documents(documents=documents, ids=uuids)
print(f"{len(documents)} chunks berhasil disimpan ke vector store")

In [ ]:
# Simpan ke disk agar tidak perlu embed ulang
vector_store.save_local("faiss_index")
print("Index tersimpan di folder faiss_index/")

In [ ]:
# State adalah "tas" yang dibawa sepanjang perjalanan graph
# Setiap node bisa membaca dan mengisi field di State
class State(TypedDict):
    question: str           # pertanyaan dari user
    context: List[Document] # hasil retrieve dari FAISS
    answer: str             # jawaban final dari LLM

In [ ]:
# Tarik prompt RAG standar dari LangChain Hub
# Prompt ini punya dua variabel: {context} dan {question}

prompt = ChatPromptTemplate.from_messages([
    ("system", """
     
Anda adalah bot customer service dan pelayan restoran Mam Sedap.

Tujuan Anda bukan hanya menjawab pertanyaan, tetapi mengundang pelanggan untuk bertanya lebih dan membuat pelanggan tertarik dengan restoran.

Gunakan bahasa Indonesia yang hangat, ramah, dan mengundang.
Berikan rekomendasi beserta alasan.
Gunakan emoji secukupnya.
Jika pelanggan meminta rekomendasi makanan, jelaskan cita rasa dan kecocokannya.
Akhiri dengan pertanyaan lanjutan yang bersedia menjawab pertanyaan pelanggan.

Jawab pelanggan sesuai dengan bahasa pelanggan.

Peraturan:
- Jangan buat kode apapun
- Jangan menjawab pertanyaan yang tidak berhubungan

Context:
{context}"""),
    ("human", "{question}")
])

# Lihat isi prompt-nya
example_messages = prompt.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

# assert len(example_messages) == 1
print("Isi RAG prompt:")
print(example_messages[0].content)

In [ ]:
llm_client = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [ ]:
# Node 1: Retrieve — ambil dokumen relevan dari FAISS
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


# Node 2: Generate — buat jawaban berdasarkan context + question
def generate(state: State):
    # Gabungkan semua chunk yang diambil jadi satu string konteks
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    # Isi template prompt dengan konteks dan pertanyaan
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    # Kirim ke LLM
    response = llm_client.invoke(messages)
    return {"answer": response.content}

In [ ]:
from langgraph.graph import START, StateGraph

# Definisikan graph: retrieve dulu, lalu generate
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

print("Graph berhasil dikompilasi!")

In [ ]:
# Visualisasi alur graph
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"question": "Rekomendasi menu?"})

print("=== Context yang diambil dari FAISS ===")
for i, doc in enumerate(result["context"]):
    print(f"\nChunk {i+1}:")
    print(doc.page_content[:200] + "...")

print("\n=== Jawaban LLM ===")
print(result["answer"])